# SatQuery AI — Master Colab Notebook

This notebook provides the fastest complete demonstration of all SatQuery AI capabilities:
- VQA (Visual Question Answering)
- Captioning
- Grounding
- Change Detection
- Optical-SAR Fusion

**No external data downloads required** for the baseline demo.

In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================
import os, sys

# Clone repo (skip if already in repo directory)
if not os.path.exists('src'):
    !git clone https://github.com/your-repo/satquery-ai .

!pip install -r requirements-colab.txt -q
print('Setup complete.')


In [ ]:
# ============================================================
# CELL 2: Check GPU and verify installation
# ============================================================
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

!python scripts/check_environment.py


In [ ]:
# ============================================================
# CELL 3: Smoke test — verify all components
# ============================================================
!python scripts/smoke_test.py --no-load


In [ ]:
# ============================================================
# CELL 4: Initialize controller (loads all models lazily)
# ============================================================
import numpy as np
import yaml
from src.agent.controller import AgentController

with open('configs/colab_8gb.yaml') as f:
    config = yaml.safe_load(f)

controller = AgentController(config, device=device)
print('Controller ready.')
print('Model status:', controller.registry.status())


In [ ]:
# ============================================================
# CELL 5: Demo — Single Image VQA
# ============================================================
# Create a synthetic test image (replace with real satellite image)
# In production, load with: from src.preprocessing.geotiff_utils import read_image
rng = np.random.RandomState(42)
# Simulate optical image with vegetation and water patches
image = rng.uniform(0.1, 0.8, (4, 64, 64)).astype('float32')
image[:, :20, :20] = 0.1  # water-like dark patch
image[3, 30:50, 30:50] = 0.7  # NIR high = vegetation

queries = [
    'What is the dominant land cover type?',
    'Is there water visible in this image?',
    'What vegetation types are present?',
]

print('=== VQA Demo ===')
for q in queries:
    result = controller.run({'image': image}, q)
    print(f'Q: {q}')
    print(f'A: {result["text"]}')
    print(f'   confidence={result["confidence"]:.2f} status={result["status"]}')
    print(f'   model={result["metadata"]["backbone"]}')
    print()


In [ ]:
# ============================================================
# CELL 6: Demo — Captioning
# ============================================================
result = controller.run({'image': image}, 'Describe the land cover visible in this satellite image.')
print('=== Captioning Demo ===')
print('Caption:', result['text'])
print('Confidence:', result['confidence'])
print('Task:', result['task'])


In [ ]:
# ============================================================
# CELL 7: Demo — Grounding
# ============================================================
result = controller.run({'image': image}, 'Highlight the water body in the image.')
print('=== Grounding Demo ===')
print('Result:', result['text'])
bbox = result['spatial_evidence']['bbox'] if result.get('spatial_evidence') else None
print('Bounding box:', bbox)
print('Source:', result['spatial_evidence']['source'] if result.get('spatial_evidence') else 'none')

# Visualize
if bbox:
    from src.models.grounding_model import visualize_bbox
    import matplotlib.pyplot as plt
    
    rgb = (image[:3].transpose(1,2,0) * 255).clip(0,255).astype('uint8')
    vis = visualize_bbox(rgb, bbox, color=(255,0,0))
    plt.figure(figsize=(5,5))
    plt.imshow(vis)
    plt.title('Grounding Result')
    plt.axis('off')
    plt.show()


In [ ]:
# ============================================================
# CELL 8: Demo — Change Detection
# ============================================================
# Simulate bi-temporal pair: T2 has a new built-up patch
t1 = rng.uniform(0.1, 0.5, (3, 64, 64)).astype('float32')
t2 = t1.copy()
t2[:, 20:40, 20:40] = 0.8  # new bright area (simulates new construction)

result = controller.run(
    {'image_t1': t1, 'image_t2': t2},
    'What changed between T1 and T2? Has the built-up area increased?'
)

print('=== Change Detection Demo ===')
print('Answer:', result['text'])
print('Confidence:', result['confidence'])
print('Model:', result['metadata']['backbone'])

# Visualize change mask
mask = result['spatial_evidence']['mask'] if result.get('spatial_evidence') else None
if mask is not None:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(t1.transpose(1,2,0)[:,:,:3])
    axes[0].set_title('T1')
    axes[1].imshow(t2.transpose(1,2,0)[:,:,:3])
    axes[1].set_title('T2')
    axes[2].imshow(mask, cmap='Reds')
    axes[2].set_title('Change Mask')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CELL 9: Demo — Optical-SAR Fusion
# ============================================================
optical = rng.uniform(0.1, 0.8, (4, 64, 64)).astype('float32')
sar = rng.uniform(0.1, 0.6, (2, 64, 64)).astype('float32')

result = controller.run(
    {'image_optical': optical, 'image_sar': sar},
    'Identify built-up areas and water bodies using both optical and SAR images.'
)

print('=== Optical-SAR Fusion Demo ===')
print('Result:', result['text'])
print('Status:', result['status'])
print('Modalities:', result['metadata']['input_modalities'])

# Show class probabilities
params = result['metadata']['parameters']
if 'class_probabilities' in params:
    probs = params['class_probabilities']
    top5 = sorted(probs.items(), key=lambda x: -x[1])[:5]
    print('\nTop-5 predicted classes:')
    for cls, prob in top5:
        print(f'  {cls}: {prob:.3f}')


In [ ]:
# ============================================================
# CELL 10: Execution Trace
# ============================================================
# Show the auditable execution trace from the last query
import json
trace = result.get('trace', {})
# Sanitize for display
trace_display = {k: v for k, v in trace.items() if k != 'mask'}
print('=== Execution Trace ===')
print(json.dumps(trace_display, indent=2, default=str))


In [ ]:
# ============================================================
# CELL 11: Metrics Demo
# ============================================================
from src.evaluation.metrics import vqa_accuracy, change_f1, caption_scores

# VQA accuracy
preds = ['Yes', 'Water body', 'Urban area']
golds = ['yes', 'Water bodies', 'Urban fabric']
acc = vqa_accuracy(preds, golds)
print(f'VQA Accuracy (normalized): {acc:.2f}')

# Caption metrics
cap_preds = ['The satellite image shows water bodies and vegetation.']
cap_golds = ['A satellite scene with water and green vegetation cover.']
scores = caption_scores(cap_preds, cap_golds)
print(f'BLEU-4: {scores["bleu4"]:.4f}  ROUGE-L: {scores["rouge_l"]:.4f}')

# Change F1
pred_mask = np.zeros((64,64), dtype=np.uint8)
pred_mask[20:40,20:40] = 1
gold_mask = np.zeros((64,64), dtype=np.uint8)
gold_mask[22:38,22:38] = 1
cf = change_f1([pred_mask], [gold_mask])
print(f'Change F1: {cf["f1"]:.4f}  IoU: {cf["iou"]:.4f}')

print('\n=== All demos complete! ===')
